# Đánh giá bản dịch và attention

TV1 tổng hợp. Mỗi lượt giải mã ghi một JSONL trong artifacts/predictions/test, mỗi dòng có id và prediction tiếng Việt đã giải mã. Tên tệp ghi rõ mô hình, tokenizer, decoder và seed.

In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/corpus.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scripts.corpus import records

sample = next(records("train"))
print(sample)

In [ ]:
import json

def score_file(path, references):
    from sacrebleu.metrics import BLEU

    with path.open(encoding="utf8") as stream:
        rows = [json.loads(line) for line in stream]
    assert [row["id"] for row in rows] == [r["id"] for r in references], "Sai ID hoặc thứ tự câu"
    metric = BLEU(tokenize="13a", lowercase=False)
    result = metric.corpus_score([r["prediction"] for r in rows], [[r["vi"] for r in references]])
    return path.stem, round(result.score, 2), str(metric.get_signature())

files = sorted((root / "artifacts/predictions/test").glob("*.jsonl"))
if files:
    refs = list(records("test"))
    for path in files:
        print(score_file(path, refs))
else:
    print("Chưa có bản dịch test để tính BLEU.")

## Trực quan hóa và đọc lỗi

Gọi hàm với trọng số trên các token thực đã cắt PAD. Với Transformer, ghi layer và head trong tiêu đề. Đối chiếu ví dụ thiếu ý, sai tên riêng, lặp từ và đảo trật tự giữa hai decoder.

In [ ]:
def plot_attention(matrix, source_tokens, target_tokens, title):
    import matplotlib.pyplot as plt

    assert matrix.shape == (len(target_tokens), len(source_tokens))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.imshow(matrix, aspect="auto", cmap="Blues")
    ax.set_xticks(range(len(source_tokens)), labels=source_tokens, rotation=90)
    ax.set_yticks(range(len(target_tokens)), labels=target_tokens)
    ax.set(xlabel="Tiếng Anh", ylabel="Tiếng Việt", title=title)
    fig.tight_layout()
    return fig